# MITRE Mitigation カバー率から `Mi` を自動算出する notebook

この notebook は、**手動アップロード**前提で次を実行します。

1. 元の Excel ファイルを 1つアップロード  
2. MITRE ATT&CK の STIX JSON を 1つアップロード  
3. 総務省ガイドライン文書を 1つアップロード  
4. デジタル庁リスク分析文書を **5ファイルまとめてアップロード**  
5. STIX から各 `Technique_ID` に対応する `Mitigation` を抽出  
6. ガイドライン文書との**意味類似度**で「カバーされている Mitigation」を判定  
7. `n_i / N_i` から対策実施率 `m_i` を算出  
8. 5段階評価 `L_i` に変換  
9. `Mi` を算出  
10. 結果を Excel に保存してダウンロード

## 算出の考え方

- `N_i` : その Technique に紐づく MITRE Mitigation 数
- `n_i` : ガイドライン文書中に意味的に対応が見つかった Mitigation 数
- `m_i = n_i / N_i`
- `L_i` : `m_i` を 5段階に変換した値
- `Mi = 1 - (1 - Mmin) * (L_i / Lmax)`

既定値:
- `Mmin = 0.1`
- `Lmax = 5`

> 注意  
> カバー判定は**意味類似度ベースの自動判定**です。  
> 厳密な実装有無の最終確認は、人手レビューを推奨します。

In [ ]:
# Colab 初回のみ必要
!pip -q install sentence-transformers pypdf openpyxl pandas

In [ ]:
import io
import json
import os
import re
from typing import List

import numpy as np
import pandas as pd
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, util

print("ライブラリを読み込みました。")

## 0. パラメータ設定
- `SIM_THRESHOLD`: Mitigation と文書文の意味類似度しきい値
- `Mmin`: 最大対策時にも残す残存リスク下限
- `LEVEL_BINS`: `m_i` を 5段階に変換する境界

In [ ]:
SIM_THRESHOLD = 0.45
Mmin = 0.1
Lmax = 5
LEVEL_BINS = [0.2, 0.4, 0.6, 0.8]

print({
    "SIM_THRESHOLD": SIM_THRESHOLD,
    "Mmin": Mmin,
    "Lmax": Lmax,
    "LEVEL_BINS": LEVEL_BINS
})

## 1. 元の Excel を 1ファイルアップロード
必須列:
- `Technique_ID`

In [ ]:
uploaded_excel = files.upload()
if len(uploaded_excel) != 1:
    raise ValueError("Excel ファイルを 1つだけアップロードしてください。")

excel_filename = list(uploaded_excel.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded_excel[excel_filename]))

print(f"Excel: {excel_filename}")
print("列名:", df.columns.tolist())
print("行数:", len(df))

if "Technique_ID" not in df.columns:
    raise ValueError("Technique_ID 列が見つかりません。")

df.head()

## 2. STIX JSON を 1ファイルアップロード
例:
- `enterprise-attack.json`

In [ ]:
uploaded_json = files.upload()
if len(uploaded_json) != 1:
    raise ValueError("STIX JSON ファイルを 1つだけアップロードしてください。")

json_filename = list(uploaded_json.keys())[0]
stix_data = json.loads(uploaded_json[json_filename].decode("utf-8"))

print(f"STIX JSON: {json_filename}")
print("objects数:", len(stix_data.get("objects", [])))

## 3. 総務省ガイドライン文書を 1ファイルアップロード
対応形式:
- PDF
- TXT
- MD

In [ ]:
uploaded_soumu = files.upload()
if len(uploaded_soumu) != 1:
    raise ValueError("総務省文書を 1つだけアップロードしてください。")

soumu_filename = list(uploaded_soumu.keys())[0]
soumu_bytes = uploaded_soumu[soumu_filename]
print(f"総務省文書: {soumu_filename}")

## 4. デジタル庁リスク分析文書を 5ファイルまとめてアップロード
対応形式:
- PDF
- TXT
- MD

ここは **5ファイル同時** にアップロードしてください。

In [ ]:
uploaded_digital = files.upload()
if len(uploaded_digital) != 5:
    raise ValueError("デジタル庁文書は 5ファイルまとめてアップロードしてください。")

digital_filenames = list(uploaded_digital.keys())
print("デジタル庁文書:")
for name in digital_filenames:
    print("-", name)

## 5. 文書読込ユーティリティ

In [ ]:
def read_text_from_uploaded_file(filename: str, file_bytes: bytes) -> str:
    lower = filename.lower()
    if lower.endswith(".pdf"):
        reader = PdfReader(io.BytesIO(file_bytes))
        pages = []
        for page in reader.pages:
            try:
                txt = page.extract_text() or ""
            except Exception:
                txt = ""
            pages.append(txt)
        return "\n".join(pages)
    elif lower.endswith(".txt") or lower.endswith(".md"):
        return file_bytes.decode("utf-8", errors="ignore")
    else:
        raise ValueError(f"未対応形式です: {filename}")

soumu_text = read_text_from_uploaded_file(soumu_filename, soumu_bytes)

digital_texts = []
for fname in digital_filenames:
    digital_texts.append(read_text_from_uploaded_file(fname, uploaded_digital[fname]))

digital_text = "\n".join(digital_texts)
combined_text = soumu_text + "\n" + digital_text

print("総務省文書文字数:", len(soumu_text))
print("デジタル庁文書合計文字数:", len(digital_text))
print("合計文字数:", len(combined_text))

## 6. STIX から Technique → Mitigation を展開

In [ ]:
objects = stix_data.get("objects", [])
attack_patterns = {}
course_of_action = {}
relationships = []

for obj in objects:
    obj_type = obj.get("type")
    if obj_type == "attack-pattern":
        if obj.get("revoked", False) or obj.get("x_mitre_deprecated", False):
            continue
        attack_id = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack" and ref.get("external_id"):
                attack_id = ref["external_id"]
                break
        if attack_id:
            attack_patterns[obj["id"]] = {
                "technique_id": attack_id,
                "name": obj.get("name", ""),
                "description": (obj.get("description", "") or "").replace("\n", " ").strip()
            }
    elif obj_type == "course-of-action":
        if obj.get("revoked", False) or obj.get("x_mitre_deprecated", False):
            continue
        mit_id = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack" and ref.get("external_id"):
                mit_id = ref["external_id"]
                break
        course_of_action[obj["id"]] = {
            "mitigation_id": mit_id,
            "name": obj.get("name", ""),
            "description": (obj.get("description", "") or "").replace("\n", " ").strip()
        }
    elif obj_type == "relationship":
        relationships.append(obj)

technique_to_mitigations = {}

for rel in relationships:
    if rel.get("relationship_type") != "mitigates":
        continue
    source_ref = rel.get("source_ref")
    target_ref = rel.get("target_ref")
    if source_ref in course_of_action and target_ref in attack_patterns:
        t_id = attack_patterns[target_ref]["technique_id"]
        mit = course_of_action[source_ref]
        technique_to_mitigations.setdefault(t_id, []).append(mit)

for t_id, mits in technique_to_mitigations.items():
    uniq = []
    seen = set()
    for m in mits:
        key = (m.get("mitigation_id"), m.get("name"), m.get("description"))
        if key not in seen:
            seen.add(key)
            uniq.append(m)
    technique_to_mitigations[t_id] = uniq

print("TechniqueにMitigationが付与された件数:", len(technique_to_mitigations))

## 7. 文書を文単位に分割

In [ ]:
def split_into_sentences(text: str):
    text = re.sub(r"\s+", " ", text)
    parts = re.split(r'(?<=[。.!?])\s+', text)
    return [p.strip() for p in parts if p and len(p.strip()) >= 15]

doc_sentences = split_into_sentences(combined_text)
print("比較対象文数:", len(doc_sentences))
doc_sentences[:5]

## 8. 埋め込みモデルをロード

In [ ]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
doc_embeddings = model.encode(
    doc_sentences,
    convert_to_tensor=True,
    show_progress_bar=True
)
print("モデル読込・文書埋め込み完了")

## 9. Technique ごとの Mitigation カバー率を算出

In [ ]:
def calc_level_from_ratio(ratio, bins):
    if ratio < bins[0]:
        return 1
    elif ratio < bins[1]:
        return 2
    elif ratio < bins[2]:
        return 3
    elif ratio < bins[3]:
        return 4
    else:
        return 5

def calc_Mi(level, Mmin=0.1, Lmax=5):
    return 1 - (1 - Mmin) * (level / Lmax)

technique_summary_rows = []
technique_mitigation_rows = []

unique_tids = sorted(df["Technique_ID"].dropna().astype(str).str.strip().unique().tolist())

for tid in unique_tids:
    mitigations = technique_to_mitigations.get(tid, [])
    N_i = len(mitigations)

    if N_i == 0:
        technique_summary_rows.append({
            "Technique_ID": tid,
            "Mitigation_Count_Ni": 0,
            "Covered_Mitigation_Count_ni": 0,
            "Coverage_Ratio_mi": np.nan,
            "Level_Li": np.nan,
            "Mi": np.nan,
            "Matched_Mitigations": "",
            "Notes": "STIX上にMitigationなし"
        })
        continue

    covered_count = 0
    matched_names = []

    for mit in mitigations:
        query = (mit.get("name", "") + " " + mit.get("description", "")).strip()
        q_emb = model.encode(query, convert_to_tensor=True)
        sims = util.cos_sim(q_emb, doc_embeddings)[0]
        best_idx = int(np.argmax(sims.cpu().numpy()))
        best_score = float(sims[best_idx].cpu().numpy())
        best_sentence = doc_sentences[best_idx]

        covered = best_score >= SIM_THRESHOLD
        if covered:
            covered_count += 1
            matched_names.append(mit.get("name", ""))

        technique_mitigation_rows.append({
            "Technique_ID": tid,
            "Mitigation_ID": mit.get("mitigation_id", ""),
            "Mitigation_Name": mit.get("name", ""),
            "Mitigation_Description": mit.get("description", ""),
            "Covered": covered,
            "Best_Similarity": round(best_score, 4),
            "Best_Matched_Sentence": best_sentence
        })

    m_i = covered_count / N_i
    L_i = calc_level_from_ratio(m_i, LEVEL_BINS)
    M_i = calc_Mi(L_i, Mmin=Mmin, Lmax=Lmax)

    technique_summary_rows.append({
        "Technique_ID": tid,
        "Mitigation_Count_Ni": N_i,
        "Covered_Mitigation_Count_ni": covered_count,
        "Coverage_Ratio_mi": round(m_i, 4),
        "Level_Li": L_i,
        "Mi": round(M_i, 4),
        "Matched_Mitigations": " | ".join(matched_names),
        "Notes": ""
    })

tech_summary_df = pd.DataFrame(technique_summary_rows)
tech_mitigation_df = pd.DataFrame(technique_mitigation_rows)

print("Technique集計件数:", len(tech_summary_df))
tech_summary_df.head()

## 10. 元Excelに `N_i / n_i / m_i / L_i / Mi` を追記

In [ ]:
result_df = df.copy()
result_df["Technique_ID"] = result_df["Technique_ID"].astype(str).str.strip()

merge_cols = [
    "Technique_ID",
    "Mitigation_Count_Ni",
    "Covered_Mitigation_Count_ni",
    "Coverage_Ratio_mi",
    "Level_Li",
    "Mi",
    "Matched_Mitigations",
    "Notes"
]

result_df = result_df.merge(
    tech_summary_df[merge_cols],
    on="Technique_ID",
    how="left"
)

result_df.head()

## 11. 保存してダウンロード
出力内容:
- `result_with_Mi` : 元Excel + Ni, ni, mi, Li, Mi
- `technique_summary` : Technique単位集計
- `mitigation_match_detail` : Mitigationごとの一致詳細

In [ ]:
base_name = os.path.splitext(excel_filename)[0]
output_filename = f"{base_name}_with_Mi_from_guidelines.xlsx"

with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    result_df.to_excel(writer, index=False, sheet_name="result_with_Mi")
    tech_summary_df.to_excel(writer, index=False, sheet_name="technique_summary")
    tech_mitigation_df.to_excel(writer, index=False, sheet_name="mitigation_match_detail")

print(f"保存完了: {output_filename}")

In [ ]:
files.download(output_filename)

## 補足
- PDFの抽出品質や文書表現差により、自動一致は完全ではありません。
- `SIM_THRESHOLD` を上げると厳しめ、下げると広めに一致します。
- `mitigation_match_detail` シートを見て、人手で最終確認してください。
- 同じ手順で別ファイルにも再利用できます。